# 00 - Setup e Configuração do Ambiente Unificado
**Squad 2 — Real Time for Business | Unificação Dupla 1 & Dupla 5 (Grupo 1 & Grupo 5)**  
**Integrantes:** Zaiden Emiliano Segundo Seleme & Lucas Sousa Santos Oliveira  
**Tabelas de Escopo:**
* **Grupo 1 (Dupla 1):** `ecommerce_produtos` e `ecommerce_categorias`
* **Grupo 5 (Dupla 5):** `ecommerce_rastreamento_entregas` e `ecommerce_enderecos`

**Branch:** `feat/squad2-dupla1e5`

---

### Objetivos:
1. Instalação automatizada das bibliotecas essenciais no ambiente Databricks Serverless.
2. Localização dinâmica e carregamento seguro das credenciais via `python-dotenv` com busca hierárquica multi-nível.
3. Validação das variáveis de ambiente para **Azure Data Lake Storage Gen2 (ADLS Gen2)** e **Azure SQL Server**.
4. Configuração das credenciais OAuth FQDN para o conector Spark ABFS (`fs.azure.account.*`).
5. Mapeamento centralizado de caminhos no Data Lake contemplando os namespaces de isolamento (`/grupo1/` e `/grupo5/`).
6. Teste de conectividade eager com o ADLS Gen2 utilizando leitor resiliente para contornar limitações de schemas (`TIMESTAMP_NANOS` e `INT32 Null`).

In [ ]:
# 1. Instalação das bibliotecas no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

In [ ]:
# 2. Reinicialização segura do interpretador Python para carregar novos pacotes
dbutils.library.restartPython()

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# 3. Localização dinâmica e carga do arquivo .env com busca resiliente
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Arquivo .env carregado com sucesso de: {dotenv_path}")
else:
    load_dotenv(override=True)
    print("Tentativa de carregar .env a partir das variáveis de ambiente padrão.")

# Leitura e validação das variáveis do ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Leitura e validação das variáveis do Azure SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"
assert sql_password, "SQL_PASSWORD não configurado no .env"

print("\n=== Status das Credenciais ADLS Gen2 ===")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID:       {'[CONFIGURADO]' if client_id else '[PENDENTE]'}")
print(f"  Tenant ID:       {'[CONFIGURADO]' if tenant_id else '[PENDENTE]'}")
print(f"  Client Secret:   {'[CONFIGURADO]' if client_secret else '[PENDENTE]'}")

print("\n=== Status das Credenciais SQL Server ===")
print(f"  SQL Host:        {sql_host}")
print(f"  SQL Database:    {sql_database}")
print(f"  SQL User:        {sql_user}")
print(f"  SQL Password:    {'[CONFIGURADO]' if sql_password else '[PENDENTE]'}")

In [ ]:
# 4. Configuração OAuth com FQDN do Storage Account para injeção granular no Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# 5. Definição da malha completa de caminhos ABFSS (Grupo 1 e Grupo 5)
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad_g1 = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"
base_squad_g5 = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo5"

caminhos = {
    # Origens no container RAW (Landing Zone)
    "raw_produtos": f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet",
    "raw_categorias": f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet",
    "raw_rastreamento": f"{base_raw}/*/*/*/*/ecommerce_rastreamento.parquet",
    "raw_enderecos": f"{base_raw}/*/*/*/*/ecommerce_enderecos.parquet",
    
    # Grupo 1 (Dupla 1) - Produtos e Categorias
    "bronze_produtos": f"{base_squad_g1}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad_g1}/bronze/ecommerce_categorias",
    "silver_produtos": f"{base_squad_g1}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad_g1}/silver/ecommerce_categorias",
    "quarantine_produtos": f"{base_squad_g1}/quarantine/ecommerce_produtos",
    "quarantine_categorias": f"{base_squad_g1}/quarantine/ecommerce_categorias",
    "gold_dim_produtos": f"{base_squad_g1}/gold/gold_dim_produtos",
    "gold_metricas_categorias": f"{base_squad_g1}/gold/gold_metricas_categorias",
    "metadata_control_g1": f"{base_squad_g1}/metadata/ingestion_control_log",

    # Grupo 5 (Dupla 5) - Rastreamento e Endereços
    "bronze_rastreamento": f"{base_squad_g5}/bronze/ecommerce_rastreamento_entregas",
    "bronze_enderecos": f"{base_squad_g5}/bronze/ecommerce_enderecos",
    "silver_rastreamento": f"{base_squad_g5}/silver/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_squad_g5}/silver/ecommerce_enderecos",
    "quarantine_rastreamento": f"{base_squad_g5}/quarantine/ecommerce_rastreamento_entregas",
    "quarantine_enderecos": f"{base_squad_g5}/quarantine/ecommerce_enderecos",
    "gold_pedidos_rota": f"{base_squad_g5}/gold/gold_logistica_pedidos_rota",
    "gold_performance_transportadoras": f"{base_squad_g5}/gold/gold_performance_transportadoras",
    "gold_distribuicao_geografica": f"{base_squad_g5}/gold/gold_distribuicao_geografica_clientes",
    "metadata_control_g5": f"{base_squad_g5}/metadata/ingestion_control_log"
}

print("Malha de caminhos configurada para Squad 2 (Grupo 1 + Grupo 5):")
for k, v in caminhos.items():
    print(f"  {k:30}: {v}")

In [ ]:
# 6. Teste de Validação de Conectividade Eager via Leitor Resiliente Serverless
import io
import pyarrow.parquet as pq
import pyarrow as pa
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa):
    cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
    service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
    fs = service.get_file_system_client("raw")
    
    paths = fs.get_paths(path="real-time-data")
    arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
    
    tables = []
    for p in arquivos:
        fc = fs.get_file_client(p)
        data = fc.download_file().readall()
        t = pq.read_table(io.BytesIO(data))
        if t.num_rows == 0 or len(t.schema.names) == 0:
            continue
        if set(schema_pa.names).issubset(set(t.schema.names)):
            t = t.select(schema_pa.names)
        t_casted = t.cast(schema_pa)
        full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
        t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
        tables.append(t_with_meta)
        
    if not tables:
        return None
    combined = pa.concat_tables(tables)
    return spark.createDataFrame(combined.to_pandas())

print("\n=== Teste de Conectividade com Origens e Destinos ===")

# Teste 1: Produtos e Categorias (Grupo 1) via Spark direto
try:
    df_prod_teste = spark.read.options(**adls_options).parquet(caminhos["raw_produtos"]).limit(5)
    df_cat_teste = spark.read.options(**adls_options).parquet(caminhos["raw_categorias"]).limit(5)
    print(f"[SUCESSO] Grupo 1: Origens de Produtos e Categorias acessíveis via Spark ABFS.")
except Exception as e_g1:
    print(f"[AVISO/FALHA] Grupo 1: {e_g1}")

# Teste 2: Rastreamento e Endereços (Grupo 5) via Leitor Resiliente
try:
    df_rast_teste = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa)
    df_end_teste = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa)
    print(f"[SUCESSO] Grupo 5: Origens de Rastreamento ({df_rast_teste.count() if df_rast_teste else 0} linhas) e Endereços ({df_end_teste.count() if df_end_teste else 0} linhas) acessíveis.")
except Exception as e_g5:
    print(f"[AVISO/FALHA] Grupo 5: {e_g5}")

def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("\nStatus das Tabelas Delta nos contêineres:")
print(f"  Bronze Produtos:     {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_produtos']) else '[NOVO]'}")
print(f"  Bronze Categorias:   {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_categorias']) else '[NOVO]'}")
print(f"  Bronze Rastreamento: {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_rastreamento']) else '[NOVO]'}")
print(f"  Bronze Endereços:    {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_enderecos']) else '[NOVO]'}")

print("\nSetup Unificado da Squad 2 concluído com sucesso!")